### Read the Bronze Sellers table

In [0]:
from pyspark.sql import functions as F

BRONZE_TABLE = "olist_mlops.bronze.sellers"

SILVER_TABLE = "olist_mlops.silver.sellers_clean"

GEO_TABLE = "olist_mlops.silver.geolocation_zip_lookup"

bronze_sellers = spark.table(BRONZE_TABLE)

print("Bronze seller rows:", bronze_sellers.count())

bronze_sellers.printSchema()

### Clean text and ZIP prefixes

In [0]:
sellers_standardized = (
    bronze_sellers

    .withColumn(
        "seller_city",
        F.trim(F.col("seller_city"))
    )

    .withColumn(
        "seller_state",
        F.upper(F.trim(F.col("seller_state")))
    )

    .withColumn(
        "seller_zip_prefix_5d",
        F.lpad(
            F.trim(F.col("seller_zip_code_prefix")),
            5,
            "0"
        )
    )
)

sellers_standardized.printSchema()

### Validate seller identifiers and ZIP prefixes

In [0]:
total_sellers = sellers_standardized.count()

null_seller_ids = sellers_standardized.filter(
    F.col("seller_id").isNull()
).count()

duplicate_seller_ids = (
    sellers_standardized
    .groupBy("seller_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_zip = sellers_standardized.filter(
    F.col("seller_zip_code_prefix").isNull()
    | ~F.trim(
        F.col("seller_zip_code_prefix")
    ).rlike(r"^[0-9]{1,5}$")
).count()

print("Total sellers:", total_sellers)
print("Null seller IDs:", null_seller_ids)
print("Duplicate seller IDs:", duplicate_seller_ids)
print("Invalid ZIP prefixes:", invalid_zip)

assert total_sellers == 3095
assert null_seller_ids == 0
assert duplicate_seller_ids == 0
assert invalid_zip == 0

print("SELLER VALIDATION PASSED")

### Prepare the lookup

In [0]:
geo_lookup = (
    spark.table(GEO_TABLE)

    .select(
        F.col("zip_prefix_5d").alias("lookup_zip"),

        F.col("latitude").alias("seller_latitude"),

        F.col("longitude").alias("seller_longitude"),

        F.col("coordinate_record_count").alias(
            "geo_coordinate_count"
        ),

        F.col("distinct_states").alias(
            "geo_distinct_states"
        )
    )
)

### Perform LEFT JOIN

In [0]:
sellers_enriched = (
    sellers_standardized

    .join(
        geo_lookup,

        sellers_standardized.seller_zip_prefix_5d
        == geo_lookup.lookup_zip,

        "left"
    )

    .withColumn(
        "location_status",
        F.when(
            F.col("lookup_zip").isNull(),
            F.lit("UNMATCHED_ZIP")
        ).otherwise(
            F.lit("MATCHED")
        )
    )

    .drop("lookup_zip")

    .withColumn(
        "silver_processed_at",
        F.current_timestamp()
    )
)

sellers_enriched.printSchema()

### Verify record counts and location coverage

In [0]:
source_count = bronze_sellers.count()

silver_count = sellers_enriched.count()

distinct_seller_count = (
    sellers_enriched
    .select("seller_id")
    .distinct()
    .count()
)

unmatched_sellers = sellers_enriched.filter(
    F.col("location_status") == "UNMATCHED_ZIP"
).count()

print("Bronze sellers:", source_count)
print("Silver sellers:", silver_count)
print("Distinct seller IDs:", distinct_seller_count)
print("Unmatched seller ZIPs:", unmatched_sellers)

assert source_count == silver_count == 3095
assert distinct_seller_count == 3095
assert unmatched_sellers == 7

print("SELLER ENRICHMENT VALIDATION PASSED")

### Examine the unmatched sellers

In [0]:
display(
    sellers_enriched
    .filter(F.col("location_status") == "UNMATCHED_ZIP")
    .select(
        "seller_id",
        "seller_zip_prefix_5d",
        "seller_city",
        "seller_state",
        "seller_latitude",
        "seller_longitude"
    )
)

### Examine location quality

In [0]:
display(
    sellers_enriched
    .groupBy("location_status")
    .count()
)

ambiguous_geo = sellers_enriched.filter(
    F.col("geo_distinct_states") > 1
).count()

print(
    "Sellers associated with ZIPs having multiple states:",
    ambiguous_geo
)

### Save the Silver Sellers Delta table

In [0]:
(
    sellers_enriched.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

assert spark.table(SILVER_TABLE).count() == 3095

print("Created:", SILVER_TABLE)

### Verify using SQL

In [0]:
%sql

SELECT
    location_status,
    COUNT(*) AS seller_count
FROM olist_mlops.silver.sellers_clean
GROUP BY location_status
ORDER BY location_status;